# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For our lane (Lane 2 — Refresh / Content Opportunity Scoring), we predict a binary target `is_declining_label` representing a $\ge 20\%$ drop in organic search impressions. We evaluated four models from the toolkit:

1. **Logistic Regression**: A linear model serving as our baseline machine learning classifier. It provides standard normalized coefficients that are highly interpretable, showing the directional sign and relative magnitude of linear relationships.
2. **Decision Tree (max_depth=5)**: A non-linear partition model. By keeping depth restricted, it remains human-readable, letting us inspect exact splits and decision thresholds.
3. **Random Forest (max_depth=10, 200 estimators)**: An ensemble method. It averages many randomized decision trees to handle non-linear feature interactions and outliers, preventing overfitting.
4. **HistGradientBoosting**: A modern gradient-boosting tree model. It fits trees sequentially to minimize remaining residuals, offering our highest-performing baseline reference.

In [1]:
# Imports and configurations
import os
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# Custom precision_at_k
def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

# Feature columns from ml_utils
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

RANDOM_STATE = 42

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Client Holdout Group Split
- **The Data Leak Trap**: The dataset contains 30,000 page-month rows from 32 distinct clients. Observations within a client domain are highly correlated (they share domain authority, industry topic niches, backlink profile, and CMS setups). If we split rows randomly, we will train and test on the same client, leading to data leakage and artificially high performance. 
- **Honest Validation**: We use a **Grouped Client Split** where we shuffle the clients and hold out 20% (6 clients out of 32) entirely for testing. We train only on the remaining 26 clients. This honestly tests whether our model generalizes to unseen client sites.

In [2]:
# Feature preparation matrix builder
def build_feature_matrix(frame):
    numeric_features = [col for col in MODEL_NUMERIC_FEATURES if col in frame.columns]
    categorical_features = [col for col in MODEL_CATEGORICAL_FEATURES if col in frame.columns]
    
    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )
    
    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)

# Client-aware split generation
def make_client_aware_split(frame, target_series):
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    
    random_generator = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = random_generator.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
    test_clients = set(shuffled_clients[:test_client_count])
    
    test_mask = client_series.isin(test_clients).to_numpy()
    train_indices = all_indices[~test_mask]
    test_indices = all_indices[test_mask]
    
    return train_indices, test_indices, test_clients

# Load prepared features dataset
df = pd.read_csv('../../data/processed/refresh_feature_vector.csv')
target_series = df["is_declining_label"].astype(int)

# Run split
train_indices, test_indices, test_clients = make_client_aware_split(df, target_series)
print("Grouped Client Split Generated:")
print(f"Held-out clients: {test_clients}")
print(f"Train rows: {len(train_indices):,}, Test rows: {len(test_indices):,}")

Grouped Client Split Generated:
Held-out clients: {'client_d4735e3a26', 'client_f74efabef1', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_1a6562590e', 'client_0b918943df'}
Train rows: 27,675, Test rows: 2,325


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# Prepare matrices
feature_frame, feature_columns = build_feature_matrix(df)
train_X = feature_frame.iloc[train_indices]
test_X = feature_frame.iloc[test_indices]
train_y = target_series.iloc[train_indices]
test_y = target_series.iloc[test_indices]

# Load our Week-4 baseline refresh scores
baseline_df = pd.read_csv('../outputs/baseline_action_score.csv')
baseline_lookup = baseline_df.set_index("content_id")["baseline_refresh_score"]
baseline_test_scores = df.iloc[test_indices]["content_id"].map(baseline_lookup).fillna(0).to_numpy()
baseline_test_probs = baseline_test_scores / 100.0 if baseline_test_scores.max() > 0 else baseline_test_scores

# Evaluation helper function
def evaluate(y_true, scores, name):
    return {
        "Model": name,
        "ROC AUC": roc_auc_score(y_true, scores),
        "Avg Precision": average_precision_score(y_true, scores),
        "Precision@10": precision_at_k(y_true, scores, 10),
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
    }

metrics_list = []

# Add baseline metrics
metrics_list.append(evaluate(test_y, baseline_test_probs, "Week-4 Baseline"))

# Initialize models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_depth=5, min_samples_leaf=25, random_state=RANDOM_STATE
    )
}

# Fit and evaluate models
for name, model in models.items():
    model.fit(train_X, train_y)
    probs = model.predict_proba(test_X)[:, 1]
    metrics_list.append(evaluate(test_y, probs, name))

# Build comparison DataFrame
metrics_df = pd.DataFrame(metrics_list)
print("=== Model Comparison Table ===")
print(metrics_df.to_string(index=False))

# Save comparison json receipt
os.makedirs('../outputs', exist_ok=True)
metrics_df.to_json('../outputs/model_comparison.json', orient='records', indent=2)

=== Model Comparison Table ===
               Model  ROC AUC  Avg Precision  Precision@10  Precision@20  Precision@50  Precision@100
     Week-4 Baseline 0.500197       0.391088           0.4          0.50          0.48           0.40
 Logistic Regression 0.700291       0.521542           0.2          0.35          0.40           0.44
       Decision Tree 0.741520       0.575319           0.7          0.55          0.62           0.60
       Random Forest 0.747372       0.610058           0.9          0.70          0.68           0.70
HistGradientBoosting 0.770966       0.683648           0.9          0.95          0.92           0.87


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Get Gini Importance for Random Forest
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
rf_imp_df = pd.DataFrame({"feature": feature_columns, "importance": importances}).sort_values("importance", ascending=False)

print("=== Top 10 Random Forest Gini Importances ===")
print(rf_imp_df.head(10).to_string(index=False))

# Compute Permutation Importance on Test Set (generalization check)
print("\n=== Calculating Permutation Importance on Test Set ===")
r = permutation_importance(rf_model, test_X, test_y, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_imp_df = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": r.importances_mean,
    "importance_std": r.importances_std
}).sort_values("importance_mean", ascending=False)

print("=== Top 10 Test Set Permutation Importances ===")
print(perm_imp_df.head(10).to_string(index=False))

=== Top 10 Random Forest Gini Importances ===
              feature  importance
  log_impressions_90d    0.131904
days_with_impressions    0.130438
         avg_position    0.111096
     content_age_days    0.090817
        age_tier_365+    0.037600
           word_count    0.036792
       log_clicks_90d    0.036497
           char_count    0.036267
   days_with_sessions    0.035031
                  ctr    0.034295

=== Calculating Permutation Importance on Test Set ===


=== Top 10 Test Set Permutation Importances ===
              feature  importance_mean  importance_std
days_with_impressions         0.035957        0.009589
  log_impressions_90d         0.010753        0.006557
       log_clicks_90d         0.008516        0.001067
   days_with_sessions         0.005849        0.000966
     content_age_days         0.004989        0.001505
  impression_tier_low         0.004731        0.002017
          scroll_rate         0.004387        0.001032
                  ctr         0.003527        0.000740
     log_sessions_90d         0.000774        0.000996
      engagement_rate         0.000430        0.000385


In [5]:
# Extract prediction errors on held-out test set
test_df_slice = df.iloc[test_indices].copy()
test_df_slice["rf_prob"] = rf_model.predict_proba(test_X)[:, 1]
test_df_slice["rf_pred"] = (test_df_slice["rf_prob"] >= 0.5).astype(int)

# Extract False Positives (high probability of decline predicted, label 0)
print("=== Top 3 False Positives ===")
fps = test_df_slice[(test_df_slice["rf_pred"] == 1) & (test_df_slice["is_declining_label"] == 0)].sort_values("rf_prob", ascending=False)
print(fps[['content_id', 'client_id', 'rf_prob', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction', 'days_with_impressions', 'content_age_days']].head(3).to_string(index=False))

# Extract False Negatives (low probability of decline predicted, label 1)
print("\n=== Top 3 False Negatives ===")
fns = test_df_slice[(test_df_slice["rf_pred"] == 0) & (test_df_slice["is_declining_label"] == 1)].sort_values("rf_prob", ascending=True)
print(fns[['content_id', 'client_id', 'rf_prob', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction', 'days_with_impressions', 'content_age_days']].head(3).to_string(index=False))

=== Top 3 False Positives ===
          content_id         client_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update trend_direction  days_with_impressions  content_age_days
content_d2dffcc697a4 client_f74efabef1 0.737431             5091          14.1 0.20                      20          stable                     88               144
content_00603b0349b4 client_f74efabef1 0.735212             1076          25.6 0.09                      20              up                     85               125
content_e55b8ab078b0 client_f74efabef1 0.733797              369          21.8 0.00                      20          stable                     63               112

=== Top 3 False Negatives ===
          content_id         client_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update trend_direction  days_with_impressions  content_age_days
content_28b4223f4e5f client_98a3ab7c34 0.081668                1           0.0  0.0                       1       

### Interpretation of Feature Importances and Errors

#### 1. Feature Reliance
The Permutation Importance and Gini Importance agree on key signals:
- `days_with_impressions` (strongest permutation impact): The number of active days receiving search console impressions. If a page has very sparse impressions, it cannot easily register a $20\%$ decline in traffic (as its baseline is already near 0). Pages with constant search presence (`days_with_impressions` close to 90) are highly vulnerable to measurable dropoffs.
- `log_impressions_90d` / `log_clicks_90d`: Total impression volume and click volume. Volume scales the capacity for decay.
- `content_age_days`: Time since publication. Newer pages exhibit higher trend volatility.

#### 2. Model Error Analysis

- **False Positives (Predicted Decline, Actually Stable/Up)**:
  - Example: `content_d2dffcc697a4` (client `client_f74efabef1`, predicted probability of decline = 73.7%, actual label = 0/stable). This page has high search presence (88 days with impressions), is relatively new (144 days), and has a very low CTR (0.20%) for its rank (avg position 14.1). The model predicts a high decline risk due to this classic decay signature, but its traffic remained stable. This indicates structural resilience in specific keywords or high-intent queries that the model cannot fully separate from standard decay candidates.
  - Example: `content_00603b0349b4` (predicted probability 73.5%, actual trend = `up`). A page with a declining signature that instead grew in search volume.

- **False Negatives (Predicted Stable, Actually Declined)**:
  - Examples: `content_28b4223f4e5f` (predicted probability 8.2%, actual trend = `down`) and `content_34b14c00f80c` (predicted probability 8.3%, actual trend = `down`).
  - The Common Cause: Both pages represent **volume-insignificant noise**. For example, `content_28b4223f4e5f` has only 1 total impression and 1 day with impressions in the 90-day feature period. Dropping from 1 impression to 0 represents a 100% decline, triggering a true label. The model correctly ignores these pages because their traffic is virtually zero (there is no real trend to model), but the label logic counts them as errors. This tells us that a minimum visibility filter (e.g. `impressions_90d >= 100` or `measurable_opportunity == 1`) is essential for a clean model target.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.